In [1]:
from dotenv import load_dotenv
import getpass
import os
from eval_utils import load_llm, load_embeddings

if not os.getenv("HUGGINGFACEHUB_API_TOKEN"):
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass("Enter your token:")

load_dotenv()

# load chat model
chat_model = load_llm("hf_models/Llama-3.1-8B-Instruct")

# load embedding model
embedding = load_embeddings(model_id="sentence-transformers/all-MiniLM-L6-v2",
                            cache_dir="hf_models/embeddings/all-MiniLM-L6-v2")

/home/a.zhang@PTW.Maschinenbau.TU-Darmstadt.de/project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.30it/s]
Device set to use cuda:0
/home/a.zhang@PTW.Maschinenbau.TU-Darmstadt.de/project/RAGAS/eval_utils.py:55: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(


In [ ]:
import os
import json
import uuid
from typing import List, Dict, Any, Optional

from pydantic import BaseModel, Field

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

from langchain_community.graphs import Neo4jGraph
from langchain_community.vectorstores import Neo4jVector

from langchain_core.retrievers import BaseRetriever

NEO4J_URI = os.environ.get("")
NEO4J_USERNAME = os.environ.get("")
NEO4J_PASSWORD = os.environ.get("")

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
)


/tmp/ipykernel_1325894/2557555067.py:23: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(


In [3]:
from eval_utils import load_docs
# load docs
docs = load_docs("data/mk4s_manuel")

100%|██████████| 8/8 [00:00<00:00, 3908.50it/s]


In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, 
                                               chunk_overlap=32,
                                               separators=["\n## ","\n### ","\n#### ", "\n\n", ".", "\n", " "],
                                               is_separator_regex=False,
                                               keep_separator=False,)
chunk_docs = []

for d in docs:
    splits = text_splitter.split_text(d.page_content)
    for i, s in enumerate(splits):
        chunk_docs.append(
            Document(
                page_content=s,
                metadata={
                    "source": d.metadata["source"],
                    "chunk_index": i,
                    "chunk_id": f"{d.metadata['source']}::chunk_{i}",
                },
            )
        )

len(chunk_docs), chunk_docs[0].metadata

(191,
 {'source': 'data/mk4s_manuel/Extruder_blob.md',
  'chunk_index': 0,
  'chunk_id': 'data/mk4s_manuel/Extruder_blob.md::chunk_0'})

Set Extraction Schema

In [5]:
ALLOWED_ENTITY_TYPES = [
    "Issue",       # Failure/Problem: Extruder blob, First layer issues, Layer shifting...
    "Symptom",     # Phenomenon: temperature jumping, layers shifted, adhesion decrease...
    "ErrorCode",   # Error code: MINTEMP, MINTEMP BED, Preheat error, Bed preheat error...
    "Component",   # Component: hotend, nozzle, thermistor, heater cartridge, heatbed, belts, pulleys, motors...
    "Setting",     # Parameter/Setting: nozzle temp, bed temp, retraction length/speed, speed percentage, power mode...
    "Material",    # Material: PLA, PETG, ABS, ASA, Nylon...
    "Tool",        # Tool: multimeter, hairdryer, heat gun, pliers, brass brush...
    "Procedure",   # Procedure/Operation flow: first layer calibration, bed leveling, belt tuning...
    "Warning",     # Safety/Warning: turn printer off, avoid short, do not overuse acetone...
]

ALLOWED_REL_TYPES = [
    "HAS_SYMPTOM",      # Issue -> Symptom
    "TRIGGERS",         # Condition/Component -> ErrorCode (simplified as Entity -> ErrorCode)
    "AFFECTS",          # Issue -> Component
    "CAUSED_BY",        # Issue/Symptom -> Component/Setting/Material/Condition
    "RESOLVED_BY",      # Issue/Symptom/ErrorCode -> Procedure/Setting/Tool/Action (expressed as Procedure)
    "REQUIRES_TOOL",    # Procedure -> Tool
    "USES_MATERIAL",    # Procedure/Issue -> Material
    "HAS_WARNING",      # Procedure/Issue -> Warning
    "RELATED_TO",       # Weak association
    "NOT_A_DEFECT",     # Issue/Symptom -> Warning/Note (e.g., Benchy hull line is a model feature)
]

In [19]:
extract_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",
         "You are an information extraction agent for an industrial on-site 3D printing operations knowledge graph.\n"
         "Goal: help frontline workers with troubleshooting, maintenance, and problem resolution.\n\n"
         "Hard requirements:\n"
         "A) You MUST always extract at least ONE entity of type Issue for every chunk.\n"
         "   - If the chunk does not explicitly name the issue, infer the primary Issue name from the document source/title.\n"
         "B) Extract actionable items whenever present: ErrorCode, Component, Setting, Procedure, Tool, Material, Warning, Symptom.\n"
         "C) Only use what is explicitly stated OR directly inferable from the source/title. Do not fabricate details.\n"
         "D) If the text states something is NOT a defect / no troubleshooting needed, use relation type NOT_A_DEFECT.\n"
         "E) Output MUST be valid JSON ONLY with keys: entities, relations.\n"
        ),
        ("human",
         "Document Source: {source}\n"
         "Chunk ID: {chunk_id}\n"
         "Text:\n{chunk_text}\n\n"
         "Return a JSON object with exactly two top-level keys:\n"
         "1) entities: a list of objects, each object has fields: id, type, name, aliases, evidence\n"
         "2) relations: a list of objects, each object has fields: source_id, type, target_id, evidence\n\n"
         "Constraints:\n"
         "- evidence must be an original snippet (<=25 words).\n"
         "- ids should be local like E1, E2, ...\n"
         "- Output JSON only (no markdown, no commentary).\n"
        ),
    ],
    template_format="f-string"
)


Entity and Relation Extraction per Document Chunk

In [20]:
def safe_parse_json(s: str):
    try:
        return json.loads(s)
    except Exception:
        return {"entities": [], "relations": []}

def run_extract_one(doc: Document, debug: bool = False) -> Dict[str, Any]:
    msg = extract_prompt.format_messages(
        source=doc.metadata["source"],
        chunk_id=doc.metadata["chunk_id"],
        chunk_text=doc.page_content
    )
    resp = chat_model.invoke(msg)
    raw = getattr(resp, "content", str(resp))

    if debug:
        print("===== RAW MODEL OUTPUT (first 1500 chars) =====")
        print(raw[:1500])
        print("==============================================")

    data = safe_parse_json(raw)
    if "entities" not in data or "relations" not in data:
        return {"entities": [], "relations": []}
    return data



In [21]:
sample = run_extract_one(chunk_docs[0], debug=True)
sample.keys(), len(sample["entities"]), len(sample["relations"])

===== RAW MODEL OUTPUT (first 1500 chars) =====
{
  "entities": [
    {
      "id": "E1",
      "type": "Issue",
      "name": "Extruder blob",
      "aliases": [],
      "evidence": "The blob (mass of plastic accumulated around the hotend) is one of the most scary-looking printing problems you might face with your 3D printer."
    },
    {
      "id": "E2",
      "type": "Component",
      "name": "Extruder",
      "aliases": [],
      "evidence": "The blob (mass of plastic accumulated around the hotend) is one of the most scary-looking printing problems you might face with your 3D printer."
    },
    {
      "id": "E3",
      "type": "Tool",
      "name": "Pliers",
      "aliases": [],
      "evidence": "carefully removing it with pliers"
    },
    {
      "id": "E4",
      "type": "Setting",
      "name": "Heating the filament",
      "aliases": [],
      "evidence": "heating the filament blob in order to soften it"
    },
    {
      "id": "E5",
      "type": "Material",
      "n

(dict_keys(['entities', 'relations']), 0, 0)

Build LLM-Extracted Triples in Neo4j (Per Chunk)

In [23]:
schema_cyphers = [
    "CREATE CONSTRAINT chunk_id_unique IF NOT EXISTS FOR (c:Chunk) REQUIRE c.chunk_id IS UNIQUE",
    "CREATE CONSTRAINT entity_uid_unique IF NOT EXISTS FOR (e:Entity) REQUIRE e.uid IS UNIQUE",
    "CREATE INDEX entity_name_index IF NOT EXISTS FOR (e:Entity) ON (e.name)",
    "CREATE INDEX entity_type_index IF NOT EXISTS FOR (e:Entity) ON (e.type)",
]

for q in schema_cyphers:
    graph.query(q)

In [24]:
def upsert_chunks(docs: List[Document], batch_size: int = 200):
    q = """
    UNWIND $rows AS row
    MERGE (c:Chunk {chunk_id: row.chunk_id})
    SET c.text = row.text,
        c.source = row.source,
        c.chunk_index = row.chunk_index
    """
    rows = [{
        "chunk_id": d.metadata["chunk_id"],
        "text": d.page_content,
        "source": d.metadata["source"],
        "chunk_index": d.metadata["chunk_index"],
    } for d in docs]

    for i in range(0, len(rows), batch_size):
        graph.query(q, params={"rows": rows[i:i+batch_size]})

upsert_chunks(chunk_docs)
"chunks upserted"


'chunks upserted'

In [ ]:
def normalize_entities_relations(extracted: Dict[str, Any]) -> Dict[str, Any]:
    ents = extracted.get("entities", []) or []
    rels = extracted.get("relations", []) or []

    local_to_uid = {}
    norm_ents = []
    for e in ents:
        etype = e.get("type")
        name = (e.get("name") or "").strip()
        if not etype or not name:
            continue
        if etype not in ALLOWED_ENTITY_TYPES:
            continue
        uid = f"{etype}::{name}".lower()
        local_to_uid[e.get("id")] = uid
        norm_ents.append({
            "uid": uid,
            "type": etype,
            "name": name,
            "aliases": e.get("aliases", []) or [],
            "evidence": e.get("evidence", "") or "",
        })

    norm_rels = []
    for r in rels:
        rt = r.get("type")
        sid = local_to_uid.get(r.get("source_id"))
        tid = local_to_uid.get(r.get("target_id"))
        if not rt or not sid or not tid:
            continue
        if rt not in ALLOWED_REL_TYPES:
            continue
        norm_rels.append({
            "source_uid": sid,
            "target_uid": tid,
            "type": rt,
            "evidence": r.get("evidence", "") or "",
        })

    return {"entities": norm_ents, "relations": norm_rels}


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (a, b, r) { ... }', position=<SummaryInputPosition line=5, column=5, offset=109>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 109, 'line': 5, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    UNWIND $rels AS r\n    MATCH (a:Entity {uid: r.source_uid})\n    MATCH (b:Entity {uid: r.target_uid})\n    CALL {\n      WITH a,b,r\n      FOREACH (_ IN CASE WHEN r.type='HAS_SYMPTOM' THEN [1] ELSE [] END |\n        MERGE (a)-[x:HAS_SYMPTOM]->(b) SET x.evidence=r.evidence\n      )\n      FOREACH (_ IN CASE WHEN r.type='TRIGGERS'

'one chunk ingested'

In [ ]:

def upsert_entities_and_relations_for_chunk(chunk_id: str, extracted: Dict[str, Any], batch_size: int = 200):
    data = normalize_entities_relations(extracted)
    ents = data["entities"]
    rels = data["relations"]

    q_ents = """
    UNWIND $ents AS e
    MERGE (n:Entity {uid: e.uid})
    SET n.type = e.type,
        n.name = e.name,
        n.aliases = e.aliases
    """
    q_mentions = """
    UNWIND $ents AS e
    MATCH (c:Chunk {chunk_id: $chunk_id})
    MATCH (n:Entity {uid: e.uid})
    MERGE (c)-[m:MENTIONS]->(n)
    SET m.evidence = e.evidence
    """

    q_rels = """
    UNWIND $rels AS r
    MATCH (a:Entity {uid: r.source_uid})
    MATCH (b:Entity {uid: r.target_uid})
    CALL {
      WITH a,b,r
      FOREACH (_ IN CASE WHEN r.type='HAS_SYMPTOM' THEN [1] ELSE [] END |
        MERGE (a)-[x:HAS_SYMPTOM]->(b) SET x.evidence=r.evidence
      )
      FOREACH (_ IN CASE WHEN r.type='TRIGGERS' THEN [1] ELSE [] END |
        MERGE (a)-[x:TRIGGERS]->(b) SET x.evidence=r.evidence
      )
      FOREACH (_ IN CASE WHEN r.type='AFFECTS' THEN [1] ELSE [] END |
        MERGE (a)-[x:AFFECTS]->(b) SET x.evidence=r.evidence
      )
      FOREACH (_ IN CASE WHEN r.type='CAUSED_BY' THEN [1] ELSE [] END |
        MERGE (a)-[x:CAUSED_BY]->(b) SET x.evidence=r.evidence
      )
      FOREACH (_ IN CASE WHEN r.type='RESOLVED_BY' THEN [1] ELSE [] END |
        MERGE (a)-[x:RESOLVED_BY]->(b) SET x.evidence=r.evidence
      )
      FOREACH (_ IN CASE WHEN r.type='REQUIRES_TOOL' THEN [1] ELSE [] END |
        MERGE (a)-[x:REQUIRES_TOOL]->(b) SET x.evidence=r.evidence
      )
      FOREACH (_ IN CASE WHEN r.type='USES_MATERIAL' THEN [1] ELSE [] END |
        MERGE (a)-[x:USES_MATERIAL]->(b) SET x.evidence=r.evidence
      )
      FOREACH (_ IN CASE WHEN r.type='HAS_WARNING' THEN [1] ELSE [] END |
        MERGE (a)-[x:HAS_WARNING]->(b) SET x.evidence=r.evidence
      )
      FOREACH (_ IN CASE WHEN r.type='RELATED_TO' THEN [1] ELSE [] END |
        MERGE (a)-[x:RELATED_TO]->(b) SET x.evidence=r.evidence
      )
      FOREACH (_ IN CASE WHEN r.type='NOT_A_DEFECT' THEN [1] ELSE [] END |
        MERGE (a)-[x:NOT_A_DEFECT]->(b) SET x.evidence=r.evidence
      )
      RETURN 1 AS ok
    }
    RETURN count(*) AS merged
    """

    if ents:
        graph.query(q_ents, params={"ents": ents})
        graph.query(q_mentions, params={"chunk_id": chunk_id, "ents": ents})
    if rels:
        graph.query(q_rels, params={"rels": rels})


test_doc = chunk_docs[0]
ex = run_extract_one(test_doc)
upsert_entities_and_relations_for_chunk(test_doc.metadata["chunk_id"], ex)
"one chunk ingested"


In [26]:
from tqdm.auto import tqdm

def build_kg_from_chunks(docs: List[Document], limit: Optional[int] = None):
    use_docs = docs[:limit] if limit else docs
    for d in tqdm(use_docs, desc="Extract+Ingest"):
        extracted = run_extract_one(d)
        upsert_entities_and_relations_for_chunk(d.metadata["chunk_id"], extracted)


build_kg_from_chunks(chunk_docs, limit=80)
"kg build (partial) done"


Extract+Ingest:   9%|▉         | 7/80 [00:56<09:52,  8.11s/it]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (a, b, r) { ... }', position=<SummaryInputPosition line=5, column=5, offset=109>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 109, 'line': 5, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    UNWIND $rels AS r\n    MATCH (a:Entity {uid: r.source_uid})\n    MATCH (b:Entity {uid: r.target_uid})\n    CALL {\n      WITH a,b,r\n      FOREACH (_ IN CASE W

'kg build (partial) done'